# VoiceForge — Phase 4: Long Documents & Audiobook Pipeline

This notebook implements the long-document pipeline with **voice consistency**, **text chunking**, and **audio concatenation**.

### Core Architecture — Voice Consistency Rule:
```
Document
   │
   ▼
Groq Document Understanding & Narration Preparation
   │
   ▼
Resolve Voice ONCE ───► Single Unified Voice Profile
   │                          │
   ▼                          │ (Applied to all chunks)
Split Narration               │
   ├── Chunk 1 ───────────────┤ ──► Audio Segment 1 ──┐
   ├── Chunk 2 ───────────────┤ ──► Audio Segment 2 ──┼──► Master Audiobook WAV
   └── Chunk N ───────────────┘ ──► Audio Segment N ──┘
```

> **Prerequisite**: Set your `GROQ_API_KEY` in Google Colab Secrets (the key icon on the left sidebar).

In [ ]:
# 1. Install Dependencies
!pip install -q groq pypdf qwen-tts soundfile torch torchaudio

In [ ]:
import os
import re
import json
import wave
import struct
import math
from pypdf import PdfReader
from groq import Groq
import soundfile as sf
from IPython.display import Audio, display

# Check Colab environment & Secrets
try:
    from google.colab import files, userdata
    IN_COLAB = True
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    IN_COLAB = False
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY not configured. Please add it to Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Initialize Qwen3-TTS VoiceDesign model
try:
    import torch
    from qwen_tts import Qwen3TTSModel
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    tts_model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map=device,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    print("Qwen3-TTS VoiceDesign model loaded successfully.")
except Exception as e:
    tts_model = None
    print(f"Notice: Qwen3-TTS not loaded ({e}). Standard fallback audio synthesizer active.")

In [ ]:
# Long Document Core Pipeline Functions

DEFAULT_VOICE = (
    "A clear, natural, warm narrator with a neutral accent, "
    "medium pace and professional delivery."
)

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b"
]

def extract_text_from_pdf(pdf_path):
    """Extract text from all pages of the document."""
    if not os.path.exists(pdf_path):
        return ""
    reader = PdfReader(pdf_path)
    pages = [p.extract_text() for p in reader.pages if p.extract_text()]
    full_text = "\n\n".join(pages)
    print(f"Extracted {len(full_text)} characters across {len(reader.pages)} pages.")
    return full_text

def prepare_narration(text):
    """Transforms raw document text into clean spoken audiobook narrative using Groq."""
    if not text or not text.strip():
        return ""
    if not client:
        return text.strip()
    
    prompt = f"""You are an expert audio scriptwriter.
Transform the following document text into a continuous, natural spoken narration script for an audiobook.
- Remove all page numbers, references, section headers, and formatting artifacts.
- Keep the narrative engaging, smooth, and comprehensive.
- Return ONLY the clean narration script text.

Text:
{text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            res = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name
            )
            narration = res.choices[0].message.content.strip()
            if narration:
                return narration
        except Exception:
            continue
    return text.strip()

def resolve_voice(full_narration_text, voice_preference=""):
    """
    CRITICAL: Resolves the voice ONCE for the entire document.
    - Preserves user explicit preferences if provided.
    - Or analyzes whole document script to determine a unified voice specification.
    """
    if voice_preference and voice_preference.strip():
        return voice_preference.strip()
    if not client:
        return DEFAULT_VOICE
        
    prompt = f"""You are a voice director. Based on the following complete audiobook narration script, determine the single ideal narrator voice profile for the entire reading.
Return ONLY a JSON object with fields:
- "voice_type": e.g. documentary_narrator, novelist, podcast_host
- "gender": male, female, neutral
- "age": young, adult, mature
- "accent": neutral, British, American, etc.
- "tone": authoritative, calm, warm, dramatic, etc.
- "emotion": neutral, dramatic, empathetic, etc.
- "pace": slow, medium, fast
- "style": documentary, storytelling, conversational
- "instruction": a concise, single-sentence voice description suitable for a TTS model.

Script Sample:
{full_narration_text[:1200]}
"""
    for model_name in GROQ_MODELS:
        try:
            res = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name,
                response_format={"type": "json_object"}
            )
            data = json.loads(res.choices[0].message.content)
            instruction = data.get("instruction")
            if instruction and isinstance(instruction, str) and instruction.strip():
                return instruction.strip()
        except Exception:
            continue
    return DEFAULT_VOICE

def split_text_into_chunks(text, max_chars=400):
    """Splits narration text into natural semantic chunks on sentence boundaries."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    current_chunk = []
    current_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if current_len + len(s) > max_chars and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [s]
            current_len = len(s)
        else:
            current_chunk.append(s)
            current_len += len(s) + 1
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def generate_chunk_audio(chunk_text, voice_instruction, output_file):
    """Synthesizes a chunk using Qwen3-TTS VoiceDesign with the unified voice instruction."""
    if tts_model is not None:
        wavs, sr = tts_model.generate_voice_design(
            text=chunk_text.strip(),
            language="English",
            instruct=voice_instruction
        )
        sf.write(output_file, wavs[0], sr)
    else:
        sample_rate = 24000
        duration = max(1.0, len(chunk_text) / 50.0)
        total_samples = int(sample_rate * duration)
        with wave.open(output_file, "w") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sample_rate)
            frames = bytearray()
            for i in range(total_samples):
                sample = int(32767 * 0.08 * math.sin(2 * math.pi * 440 * i / sample_rate))
                frames.extend(struct.pack("<h", sample))
            wav_file.writeframes(frames)

def concatenate_audio_files(chunk_files, output_file, pause_duration=0.3):
    """Seamlessly concatenates multiple audio chunk WAVs into a single master audiobook."""
    if not chunk_files:
        return False
    with wave.open(chunk_files[0], "rb") as first_wav:
        params = first_wav.getparams()
        sample_rate = params.framerate
        sampwidth = params.sampwidth
        nchannels = params.nchannels
    silence_samples = int(sample_rate * pause_duration)
    silence_frames = bytearray(silence_samples * sampwidth * nchannels)
    with wave.open(output_file, "wb") as outfile:
        outfile.setparams(params)
        for idx, f in enumerate(chunk_files):
            with wave.open(f, "rb") as infile:
                outfile.writeframes(infile.readframes(infile.getnframes()))
            if idx < len(chunk_files) - 1:
                outfile.writeframes(silence_frames)
    return True

def run_audiobook_pipeline(pdf_path, voice_preference="", output_file="master_audiobook.wav"):
    """End-to-End Long Document Audiobook Pipeline."""
    print(f"=== Processing Document: {pdf_path} ===")
    raw_text = extract_text_from_pdf(pdf_path)
    if not raw_text:
        print("No text extracted. Aborting.")
        return
        
    print("\n--- Step 1: Groq Narration Preparation ---")
    narration_script = prepare_narration(raw_text)
    print(f"Prepared Narration: {len(narration_script)} chars.")
    
    print("\n--- Step 2: Resolving Single Voice Profile (Once for entire document) ---")
    unified_voice = resolve_voice(narration_script, voice_preference)
    print(f"Unified Voice Instruction: {unified_voice}")
    
    print("\n--- Step 3: Splitting into Chunks ---")
    chunks = split_text_into_chunks(narration_script, max_chars=400)
    print(f"Generated {len(chunks)} chunks.")
    
    print("\n--- Step 4: Sequential Audio Generation ---")
    chunk_files = []
    for i, chunk in enumerate(chunks, 1):
        c_file = f"chunk_{i}.wav"
        chunk_files.append(c_file)
        print(f"Generating chunk {i}/{len(chunks)} ({len(chunk)} chars) with unified voice...")
        generate_chunk_audio(chunk, unified_voice, c_file)
        
    print(f"\n--- Step 5: Assembling Final Master Audiobook ---")
    concatenate_audio_files(chunk_files, output_file)
    print(f"Saved final audiobook to: {output_file}")
    
    for f in chunk_files:
        if os.path.exists(f):
            os.remove(f)
            
    display(Audio(output_file))
    print("Audiobook generation complete!")

### PDF Upload

In [ ]:
pdf_file = None
if IN_COLAB:
    print("Upload your PDF document:")
    uploaded = files.upload()
    if uploaded:
        pdf_file = list(uploaded.keys())[0]
else:
    pdf_file = "sample_document.pdf"
    print(f"Using sample document: {pdf_file}")

### TEST 1: Long Document with AI Voice Discovery
Groq infers the ideal unified narrator voice once for the entire document, applies it to every chunk, and concatenates the audio segments.

In [ ]:
if pdf_file:
    run_audiobook_pipeline(pdf_file, voice_preference="", output_file="audiobook_ai_voice.wav")

### TEST 2: Long Document with Explicit User Voice
Preserves the user-designed voice profile across all chunks.

In [ ]:
custom_narrator = "Deep authoritative Scottish male narrator, measured pace, dramatic and solemn."
if pdf_file:
    run_audiobook_pipeline(pdf_file, voice_preference=custom_narrator, output_file="audiobook_custom_voice.wav")